In [13]:
# DEPENDENCY MANAGEMENT (Updated for 2026 Compatibility)
!pip install -q -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    faiss-cpu \
    python-dotenv \
    keyring
print("[System] Modern Enterprise environment successfully installed.")

[System] Modern Enterprise environment successfully installed.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pathlib import Path
from dotenv import load_dotenv


print("="*60)
print(" INITIATING LCEL RAG PIPELINE ARCHITECTURE ")
print("="*60)

 INITIATING LCEL RAG PIPELINE ARCHITECTURE 


In [21]:

# Load .env (development) into env vars; production should set real env vars or use a secrets manager.
load_dotenv()  # reads .env if present

# Prefer explicit env var; fall back to a secrets file only if provided
api_key = os.getenv("GEMINI_API_KEY") 

if not api_key:
    secrets_path = Path(os.getenv("SECRETS_PATH", Path("secrets") / "api"))
    if secrets_path.exists():
        with secrets_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("GEMINI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment or secrets file. Set GEMINI_API_KEY or SECRETS_PATH.")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded:", bool(api_key))

API key loaded: True


In [22]:
# Read and sanitize DB_DIR environment value
db_env = os.getenv("DB_DIR", "").strip()
if db_env.startswith(("r\"", "r'", "R\"", "R'")):
    db_env = db_env[1:]
db_env = db_env.strip().strip('\'"')

if db_env:
    DB_DIR = Path(db_env)
else:
    app_base = Path(os.getenv("APPDATA") or os.getenv("LOCALAPPDATA") or Path.home())
    DB_DIR = app_base / "ai-foundations-lab" / "faiss_extract_once_db"

DB_DIR = DB_DIR.expanduser()

# Safe mkdir (this will fail only if DB_DIR contains invalid characters)
DB_DIR.mkdir(parents=True, exist_ok=True)


In [23]:
# 2. Initialize the Embedding Model (The Translator)
embedder = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 3. Initialize the Gemini LLM (The Reader/Writer)
# CRITICAL MLOPS CONFIGURATION: We set temperature to 0.0.
# In a RAG pipeline, we want STRICT factual answers. We do not want the AI to be
# "creative" with our corporate data. Creativity leads to hallucinations.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

# 4. Mount the FAISS Database from Disk
# We target the directory we created in Session 2.2.

print(f"\n[System] Mounting FAISS Vector Database from ./{DB_DIR} ...")
vectorstore = FAISS.load_local(
    folder_path=DB_DIR,
    embeddings=embedder,
    allow_dangerous_deserialization=True # Safe because we generated this .pkl file locally
)

# 5. Convert the Database into a LangChain Retriever
# We configure it to return only the top 3 most mathematically relevant chunks.
# This keeps the context window small, mitigating the "Lost in the Middle" phenomenon.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("[System] Retriever module initialized successfully.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



[System] Mounting FAISS Vector Database from ./C:\Users\Public\Documents\ai-foundations-lab\faiss_extract_once_db ...
[System] Retriever module initialized successfully.


In [24]:
template = """
You are an elite MLOPS auditing algorithm for the 'Extract Once, Judge Many' project.
Your core directive is to answer the user's query using STRICTLY the context provided below. 

CRITICAL RULES:
1. Anti-Hallucination Protocol: If the answer is not explicitly contained in the context, you must output: I cannot answer this based on the provided corporate documents." Do not attempt to guess. 
2. Do not use your pre-trained internet knowledge.
3. Keep your answer professional, highly concise, and cite the author if available.


=======================
RETRIEVED CONTEXT:
{context}
=======================

USER QUERY: {question}

FINAL ANSWER:
"""

# We compliel the string into a Langchain prompt object 
# Langchain automatically detects the {context} and {question} variables.
rag_prompt = PromptTemplate.from_template(template)

In [26]:
# The context formatter 
#The retriever returns a list of complex objects. We need pure text for the prompt. 
# This helper function extracts the `page_content` from each document and joins them with double newlines.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("[System] LCEL Pipeline constructed and ready for invocation.")

[System] LCEL Pipeline constructed and ready for invocation.


In [27]:
user_question = "what is the currect OCR failure rate, and what division is it impacting?"

print("=" * 60)
print(f" USER QUERY: {user_question}")
print("="*60)
print ("Processing RAG Invocation...(Translating -> Searching -> Grounding -> Generating)\n")

final_response = rag_chain.invoke(user_question)

print("==== AI Response =====")
print(final_response)
print("==================")

 USER QUERY: what is the currect OCR failure rate, and what division is it impacting?
Processing RAG Invocation...(Translating -> Searching -> Grounding -> Generating)

==== AI Response =====
The current OCR failure rate is 15% of scanned PDF documents, as noted by Marten. I cannot answer what division it is impacting based on the provided corporate documents.


In [28]:
# 10. The Anti-Hallucination Test (The Acid Test)
# We must prove our prompt guardrails work. We will ask a question about something
# that does NOT exist in our FAISS database.
print("\n[System] Executing Anti-Hallucination protocol test...")

trick_question = "What is the budget for the new Marketing campaign in Q2?"

print(f"\nTrick Query: {trick_question}")
print("Processing RAG invocation...\n")

trick_response = rag_chain.invoke(trick_question)

print(f"=== AI RESPONSE ===")
print(trick_response)
print("===================")
print("\n-> ANALYSIS: Notice how the AI explicitly refuses to answer! Because 'Marketing' is not in our FAISS database, the Retriever fetched irrelevant chunks. The LLM read those chunks, realized the answer wasn't there, and triggered our safety protocol. You have successfully defeated hallucination.")


[System] Executing Anti-Hallucination protocol test...

Trick Query: What is the budget for the new Marketing campaign in Q2?
Processing RAG invocation...

=== AI RESPONSE ===
I cannot answer this based on the provided corporate documents.

-> ANALYSIS: Notice how the AI explicitly refuses to answer! Because 'Marketing' is not in our FAISS database, the Retriever fetched irrelevant chunks. The LLM read those chunks, realized the answer wasn't there, and triggered our safety protocol. You have successfully defeated hallucination.


In [29]:
# 10. The Anti-Hallucination Test (The Acid Test)
# We must prove our prompt guardrails work. We will ask a question about something
# that does NOT exist in our FAISS database.
print("\n[System] Executing Anti-Hallucination protocol test...")

trick_question = "What is the capital of India?"

print(f"\nTrick Query: {trick_question}")
print("Processing RAG invocation...\n")

trick_response = rag_chain.invoke(trick_question)

print(f"=== AI RESPONSE ===")
print(trick_response)
print("===================")
print("\n-> ANALYSIS: Notice how the AI explicitly refuses to answer! Because 'Marketing' is not in our FAISS database, the Retriever fetched irrelevant chunks. The LLM read those chunks, realized the answer wasn't there, and triggered our safety protocol. You have successfully defeated hallucination.")


[System] Executing Anti-Hallucination protocol test...

Trick Query: What is the capital of India?
Processing RAG invocation...

=== AI RESPONSE ===
I cannot answer this based on the provided corporate documents.

-> ANALYSIS: Notice how the AI explicitly refuses to answer! Because 'Marketing' is not in our FAISS database, the Retriever fetched irrelevant chunks. The LLM read those chunks, realized the answer wasn't there, and triggered our safety protocol. You have successfully defeated hallucination.


In [30]:
rag_prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are an elite MLOPS auditing algorithm for the \'Extract Once, Judge Many\' project.\nYour core directive is to answer the user\'s query using STRICTLY the context provided below. \n\nCRITICAL RULES:\n1. Anti-Hallucination Protocol: If the answer is not explicitly contained in the context, you must output: I cannot answer this based on the provided corporate documents." Do not attempt to guess. \n2. Do not use your pre-trained internet knowledge.\n3. Keep your answer professional, highly concise, and cite the author if available.\n\n\n=======================\nRETRIEVED CONTEXT:\n{context}\n=======================\n\nUSER QUERY: {question}\n\nFINAL ANSWER:\n')